# Qabas — process the demo lessons

Runs the **real Qabas pipeline** (the same code as the app) on your lessons and saves each one as a small file,
`backend/data/seed/<name>.json`. Those files are put in the repo, and the live app loads them at startup, so visitors
open real lessons **without spending any Gemini requests**.

What one lesson does (10 minutes of audio):
1. **Transcription, two readings** + re-listening at the spots where they disagree (3.8 Flash: 2 to 6 requests)
2. Quran verification against the Mushaf (local, free)
3. **Hadith search:** Sahih al-Bukhari + Sahih Muslim (BM25 → quote rule → Flash-Lite check → Dorar)
4. Summary and review cards (Flash-Lite: its own quota)

**Free-tier budget:** about 20 requests a day for 3.8 Flash, so **one 10-minute lesson per day** is safe.
If 3.8 Flash is overloaded (503 "high demand") or its daily limit is hit, the app moves to another Flash model
(own quota) and says so in the quality note. It never transcribes with Flash-Lite because of busy servers.

## Steps
1. Colab key icon (Secrets) → `GEMINI_API_KEY` must be there with **Notebook access** on (already set from before).
2. Run **section 1** → choose the file **`qabas-prototype.zip`** when asked.
3. Edit the `LESSONS` list in **section 3** (the first one is already filled in), then run sections 3 and 4.
4. **Section 5** downloads `seed_lessons.zip`: send it back to me.

## 1. Upload the app and install

In [ ]:
from google.colab import files                        # file upload / download in Colab
import zipfile, pathlib, os

up = files.upload()                                    # choose qabas-prototype.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall("app_src")                            # unpack next to the notebook
BACKEND = next(pathlib.Path("app_src").rglob("backend")).resolve()   # absolute: the lesson runs in another folder   # the folder that contains app/ and requirements.txt
print("backend folder:", BACKEND)
!pip -q install -r {BACKEND}/requirements.txt
!ffmpeg -version | head -n 1

## 2. Key

In [ ]:
from google.colab import userdata                      # Colab's Secrets panel

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")   # read the secret (never paste the key)
os.environ["MAX_LESSON_MINUTES"] = "0"                 # the 10-minute cap is set per lesson below, not by the live-app rule
os.environ["HAYSTACK_TELEMETRY_ENABLED"] = "False"
os.environ["SEED_DIR"] = str(BACKEND / "data" / "seed")
os.environ["ALLOW_LIGHT_TRANSCRIPTION"] = "0"         # demo lessons are never transcribed by Flash-Lite (2-3x more errors)
print("Key loaded ✔")

# Which models can this key use? 3.8 Flash is the main model; other numbered Flash models are the backups.
import sys
sys.path.insert(0, str(BACKEND))
from google import genai
from app.ai.gemini import pick_backups
_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])   # kept in a variable: a client used inline gets closed mid-list
_names = sorted(m.name.split("/")[-1] for m in _client.models.list() if "gemini" in m.name)
print("Models this key can use:", ", ".join(_names))
_backups = pick_backups(_names, "gemini-3.8-flash")
print("main model 3.8 Flash available:", "gemini-3.8-flash" in _names, "| backups:", _backups or "none")

## 3. Your lessons

One line per lesson. Fields: `name` (file name), `url` (YouTube link), `book`, `title`, `sheikh`, `number`, `minutes`.
Start with one; add more on later days.

`MODEL` is normally left empty (the app uses 3.8 Flash, and moves to a backup Flash model by itself if 3.8 Flash is overloaded).
Only set it if you want to force a model from the list printed in section 2.

In [ ]:
MODEL = ""          # empty = the app's default main model (3.8 Flash); see the note above

LESSONS = [
    # 1) clip01: we have a hand-made reference transcript for it, so section 5b measures the app's accuracy
    dict(name="clip01-shuwair-takhsis-bil-shart",
         url="https://www.youtube.com/watch?v=mpso_JoPE6M",
         book="المختصر في أصول الفقه على مذهب الإمام أحمد بن حنبل، لابن اللحام",
         title="التخصيص بالشرط",
         sheikh="الشيخ عبد السلام بن محمد الشويعر",
         number=1, minutes=10),
    # 2) the demo lesson
    dict(name="sindi-usul-thalatha",
         url="https://youtu.be/pjFst8J5hTo",
         book="الأصول الثلاثة، للإمام المجدد محمد بن عبد الوهاب التميمي الحنبلي",
         title="شرح الأصول الثلاثة",
         sheikh="الشيخ صالح سندي",
         number=1, minutes=10),
]

## 3b. Diagnose: is it Google, the request, or the video?

Sends a few tiny test requests to 3.8 Flash (about $0.01 in total on the paid key):
text only, then a 20-second piece of each lesson's video in different ways.
Copy the whole output to Claude.

In [ ]:
from google.genai import types
import time, re

M = MODEL or "gemini-3.8-flash"

def _try_once(label, contents, cfg=None):
    t = time.time()
    try:
        r = _client.models.generate_content(model=M, contents=contents, config=cfg)
        print(f"  ✔ {label:42s} {time.time() - t:5.1f}s  {(r.text or '').strip()[:60]!r}")
    except Exception as e:
        print(f"  ✗ {label:42s} {time.time() - t:5.1f}s  {str(e)[:150]}")

def _video(url, offsets=True):
    vid = re.search(r"(?:v=|youtu\.be/|/live/)([A-Za-z0-9_-]{11})", url).group(1)
    meta = types.VideoMetadata(start_offset="0s", end_offset="20s") if offsets else None
    return types.Part(file_data=types.FileData(file_uri=f"https://www.youtube.com/watch?v={vid}"), video_metadata=meta)

ASK = types.Part(text="اكتب ما يقوله المتكلم في سطر واحد.")
LOW = types.GenerateContentConfig(media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW)
JSON_LOW = types.GenerateContentConfig(media_resolution=types.MediaResolution.MEDIA_RESOLUTION_LOW,
                                       response_mime_type="application/json",
                                       response_schema={"type": "OBJECT", "properties": {"text": {"type": "STRING"}}})

print("model:", M)
_try_once("text only", "Reply with the single word: yes")
for L in LESSONS:
    print("\n" + L["name"])
    _try_once("video 0-20s, low resolution (as the app)", types.Content(parts=[_video(L["url"]), ASK]), LOW)
    _try_once("video 0-20s, low res + JSON (as the app)", types.Content(parts=[_video(L["url"]), ASK]), JSON_LOW)
    _try_once("video 0-20s, default resolution", types.Content(parts=[_video(L["url"]), ASK]))
    _try_once("whole video, no offsets, low res", types.Content(parts=[_video(L["url"], False),
              types.Part(text="ما عنوان هذا الدرس؟ أجب في سطر واحد.")]), LOW)

## 4. Process (one lesson takes about 3 to 6 minutes)

In [ ]:
import subprocess, shlex, sys, json, time

for L in LESSONS:
    out = BACKEND / "data" / "seed" / f"{L['name']}.json"
    if out.exists():
        _used = json.loads(json.loads(out.read_text(encoding="utf-8"))["lesson"]["quality_json"] or "{}").get("models_used", {})
        if not any("lite" in m for m in _used):
            print("already done:", L["name"]); continue
        out.rename(out.with_suffix(".flashlite.bak"))        # made by Flash-Lite on a bad day: do it again properly
        print("redoing", L["name"], "(the earlier file was transcribed by Flash-Lite)")
    print(f"━━━━ {L['name']} ━━━━  (model: {MODEL or 'default'})")
    cmd = [sys.executable, "-m", "app.seed", "process", "--url", L["url"], "--book", L["book"], "--title", L["title"],
           "--sheikh", L["sheikh"], "--number", str(L["number"]), "--minutes", str(L["minutes"]), "--name", L["name"]]
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=BACKEND, env={**os.environ, "DATABASE_URL": "sqlite:///seed_work.db", **({"GEMINI_MODEL": MODEL} if MODEL else {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:                              # show progress as it happens
        print(line.rstrip())
    p.wait()
    print(f"time for this lesson: {(time.time() - t0) / 60:.1f} min")
    if p.returncode != 0:
        print("\n⚠ This lesson was not saved. Read 'WHAT HAPPENED' above, then run this cell again when it says so.")

## 4b. Hadith library (nine books) and the full hadith for the saved lessons

1. Downloads nine hadith books (the two Sahihs, the four Sunan, the Muwatta, the Nawawi and Qudsi Forty) from
   fawazahmed0/hadith-api (Unlicense) and builds the app's search file `hadith_corpus.json.gz`.
2. Looks up every hadith the sheikh quoted in the saved lessons again, **without re-transcribing**
   (only a few cheap Flash-Lite checks). Safe to run as many times as you like.

In [ ]:
import subprocess, sys

import gzip, json
corpus = BACKEND / "data" / "hadith_corpus.json.gz"
books = json.load(gzip.open(corpus, "rt", encoding="utf-8")).get("books", []) if corpus.exists() else []
if len(books) >= 9:
    print("hadith library already has", len(books), "books:", ", ".join(books))
else:                                                   # build it (downloads the nine books, about a minute)
    p = subprocess.run([sys.executable, "-m", "tools.build_hadith_corpus"], cwd=BACKEND, capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-2000:])

p = subprocess.run([sys.executable, "-c", "import resource as r; from app import hadith_index; "
                    "i = hadith_index.get_index(); print('memory with the index loaded:', r.getrusage(r.RUSAGE_SELF).ru_maxrss // 1024, 'MB')"],
                   cwd=BACKEND, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])

p = subprocess.run([sys.executable, "-m", "app.seed", "rehadith"], cwd=BACKEND, capture_output=True, text=True,
                   env={**os.environ, "DATABASE_URL": "sqlite:///seed_work.db"})
print(p.stdout[-8000:])
print(p.stderr[-3000:])

## 5. Look at the result, then download

Check: the number of verified verses, the hadith found (with their references), and the «غير مؤكد» spots.

In [ ]:
import json

if not list((BACKEND / "data" / "seed").glob("*.json")):
    raise SystemExit("No lesson was produced (see the error in the cell above). Nothing to download: wait a few minutes and re-run the previous cell.")

for f in sorted((BACKEND / "data" / "seed").glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    q = json.loads(d["lesson"]["quality_json"])
    print(f"━━━━ {f.name}: {d['lesson']['title']} | {len(d['segments'])} segments, {len(d['cards'])} cards")
    print("  quality:", {k: v for k, v in q.items() if k != "notes"})
    for n in q.get("notes", []): print("  note:", n)
    for s in d["segments"]:
        if s["kind"] == "quran" and s["quran_json"]:
            v = json.loads(s["quran_json"]); print(f"  آية {'✔' if v.get('verified') else '?'}  {v.get('surah_name')} {v.get('ayah_from')}: {s['text'][:60]}")
        if s["kind"] == "hadith":
            h = json.loads(s["hadith_json"]) if s["hadith_json"] else None
            print("  حديث:", s["text"][:60], "→", (h["results"][0]["source"] + " [" + h["method"] + "]") if h and h["results"] else "no full hadith")
    print("  summary:")
    for p_ in json.loads(d["lesson"]["summary_json"]): print("   -", p_["point"])

with zipfile.ZipFile("seed_lessons.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in (BACKEND / "data" / "seed").glob("*.json"):
        z.write(f, f.name)
    corpus = BACKEND / "data" / "hadith_corpus.json.gz"           # the nine-book library built in section 4b
    if corpus.exists():
        z.write(corpus, corpus.name)
files.download("seed_lessons.zip")

## 5b. Accuracy on clip01 (the app's own output vs. our hand-made reference)

Same scoring as the evaluation notebooks: harakat and punctuation removed, alef/ya/ta marbuta unified.
Earlier evaluation of the same method: **3.7% WER (plain) / 4.4% (with context), 36 of 36 terms right**.

In [ ]:
!pip -q install jiwer
import json, sys
NB = BACKEND.parent / "notebooks"
sys.path.insert(0, str(NB))
import eval_core

f = BACKEND / "data" / "seed" / "clip01-shuwair-takhsis-bil-shart.json"
if not f.exists():
    raise SystemExit("clip01 was not processed yet (see section 4).")
d = json.loads(f.read_text(encoding="utf-8"))
segs = sorted(d["segments"], key=lambda s: s["start"])
hyp = " ".join(s["text"] for s in segs if s["kind"] != "audience")      # the app never keeps audience speech
ref = (NB / "clip01" / "reference.txt").read_text(encoding="utf-8")
terms = eval_core.load_terms(str(NB / "clip01" / "terms.txt"))

r = eval_core.score_all(ref, hyp, terms)
q = json.loads(d["lesson"]["quality_json"] or "{}")
print(f"WER {r['WER']:.1%} | CER {r['CER']:.1%} | terms {r['terms_unique']} ({r['terms_occ_pct']:.0%} of occurrences)")
print("missed terms:", r["missed_terms"] or "none")
print("models used:", q.get("models_used") or "main model only", "| «غير مؤكد» marks:", q.get("uncertain"))
print("\nmost common word swaps (reference -> app):")
for (a, b), n in eval_core.top_substitutions(ref, hyp, 15):
    print(f"  {n}×  {a} -> {b}")

## 6. Measure the hadith search (111 test questions)

Runs the app's own hadith pipeline on the test set: BM25 over the nine books → quote rule → Flash-Lite check.
About 16 Flash-Lite requests (a few cents). Copy the whole output to Claude.

In [ ]:
import subprocess, sys
p = subprocess.run([sys.executable, str(BACKEND.parent / "notebooks" / "hadith_eval" / "eval_hadith.py"), "full"],
                   cwd=BACKEND, capture_output=True, text=True)
print(p.stdout[-12000:])
print(p.stderr[-3000:])
files.download(str(BACKEND.parent / "notebooks" / "hadith_eval" / "full_results.json"))